<a href="https://colab.research.google.com/github/solitude-ui/CIFAR-100-Image-Classification/blob/main/Assesment_CIFAR100_CNN_VGG16_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIML Computer Vision Case Study
## Training and Evaluating CNN Models on CIFAR-100

**Objective:** Train, evaluate, and compare a custom CNN with a VGG-16-style architecture using CIFAR-100.

>  The VGG-16 model is adapted for CIFAR-100's 32×32 RGB images to keep training more practical. It retains the 13 convolutional layers and five convolutional blocks; its classifier uses global average pooling rather than the original large 4096-unit fully connected layers.

## 1. Load libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import plot_model

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Load CIFAR-100 dataset

In [ ]:
# CIFAR-100 has 50,000 training images and 10,000 test images.
# Each image is 32×32 with 3 RGB channels; labels are integers from 0 to 99.
(X_train, y_train), (X_test, y_test) = cifar100.load_data(label_mode="fine")

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape, X_test.dtype)
print("y_test:", y_test.shape)
print("Number of classes:", len(np.unique(y_train)))

In [ ]:
# Display a sample RGB image
plt.figure(figsize=(3, 3))
plt.imshow(X_train[5])
plt.title(f"Example CIFAR-100 label: {int(y_train[5][0])}")
plt.axis("off")
plt.show()

## 3. Preprocess the data

In [ ]:
# Convert pixel values from [0, 255] to [0, 1].
# Keep RGB shape (32, 32, 3); do NOT add an extra channel dimension.
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Integer labels are compatible with sparse_categorical_crossentropy.
# Flatten labels from shape (N, 1) to (N,).
y_train = y_train.squeeze().astype("int32")
y_test = y_test.squeeze().astype("int32")

print("Processed image shape:", X_train.shape)
print("Label range:", y_train.min(), "to", y_train.max())

## 4. Define the custom CNN model

This model includes batch normalization, dropout, L2 regularization, and He weight initialization. These provide the requested opportunities to experiment with architecture and regularization.

In [ ]:
def build_custom_cnn():
    inputs = layers.Input(shape=(32, 32, 3))

    x = layers.Conv2D(
        32, 3, padding="same", kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4), use_bias=False
    )(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(1e-4), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(1e-4), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(1e-4), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(1e-4), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, kernel_initializer="he_normal",
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(100, activation="softmax")(x)

    return models.Model(inputs, outputs, name="Custom_CNN")

custom_cnn = build_custom_cnn()
custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
custom_cnn.summary()

## 5. Custom CNN architecture diagram

In [ ]:
plot_model(
    custom_cnn,
    to_file="custom_cnn_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    dpi=100
)
display(plt.imread("custom_cnn_architecture.png"))

## 6. Train the custom CNN

`ModelCheckpoint` saves the model with the best validation accuracy. `EarlyStopping` stops training when validation loss does not improve.

In [ ]:
os.makedirs("checkpoints", exist_ok=True)

custom_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath="checkpoints/best_custom_cnn.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

history_custom = custom_cnn.fit(
    X_train, y_train,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    callbacks=custom_callbacks,
    verbose=1
)

## 7. Evaluate the custom CNN on the test set

In [ ]:
custom_test_loss, custom_test_accuracy = custom_cnn.evaluate(
    X_test, y_test, verbose=0
)
print(f"Custom CNN test loss: {custom_test_loss:.4f}")
print(f"Custom CNN test accuracy: {custom_test_accuracy:.4%}")

## 8. Define the VGG-16 architecture

VGG-16 contains 13 convolutional layers arranged in five blocks, followed by three classifier layers. This CIFAR-100-friendly version uses global average pooling and a 512-unit dense layer instead of the original 4096-unit dense layers, reducing memory and computation.

In [ ]:
def build_vgg16_cifar100():
    inputs = layers.Input(shape=(32, 32, 3))
    x = inputs

    # Block 1: 2 convolutional layers
    for _ in range(2):
        x = layers.Conv2D(64, 3, padding="same", activation="relu",
                          kernel_initializer="he_normal")(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 2: 2 convolutional layers
    for _ in range(2):
        x = layers.Conv2D(128, 3, padding="same", activation="relu",
                          kernel_initializer="he_normal")(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 3: 3 convolutional layers
    for _ in range(3):
        x = layers.Conv2D(256, 3, padding="same", activation="relu",
                          kernel_initializer="he_normal")(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 4: 3 convolutional layers
    for _ in range(3):
        x = layers.Conv2D(512, 3, padding="same", activation="relu",
                          kernel_initializer="he_normal")(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 5: 3 convolutional layers
    for _ in range(3):
        x = layers.Conv2D(512, 3, padding="same", activation="relu",
                          kernel_initializer="he_normal")(x)
    x = layers.MaxPooling2D(2)(x)

    # CIFAR-100 classifier head (adapted to reduce parameters)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu", kernel_initializer="he_normal")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(100, activation="softmax")(x)

    return models.Model(inputs, outputs, name="VGG16_CIFAR100")

vgg_16_model = build_vgg16_cifar100()
vgg_16_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
vgg_16_model.summary()

## 9. VGG-16 architecture diagram

In [ ]:
plot_model(
    vgg_16_model,
    to_file="vgg16_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    dpi=100
)
display(plt.imread("vgg16_architecture.png"))

## 10. Train VGG-16

In [ ]:
vgg_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath="checkpoints/best_vgg16_cifar100.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

history_vgg16 = vgg_16_model.fit(
    X_train, y_train,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    callbacks=vgg_callbacks,
    verbose=1
)

## 11. Evaluate VGG-16 on the test set

In [ ]:
vgg_test_loss, vgg_test_accuracy = vgg_16_model.evaluate(
    X_test, y_test, verbose=0
)
print(f"VGG-16 test loss: {vgg_test_loss:.4f}")
print(f"VGG-16 test accuracy: {vgg_test_accuracy:.4%}")

## 12. Plot and compare training curves

The curves below compare training and validation accuracy/loss for each model. The test set is reserved for final evaluation rather than used during training.

In [ ]:
def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(history.history["accuracy"], label="Training accuracy")
    axes[0].plot(history.history["val_accuracy"], label="Validation accuracy")
    axes[0].set_title(f"{model_name}: Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history.history["loss"], label="Training loss")
    axes[1].plot(history.history["val_loss"], label="Validation loss")
    axes[1].set_title(f"{model_name}: Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

plot_training_history(history_custom, "Custom CNN")
plot_training_history(history_vgg16, "VGG-16")

## 13. Final test accuracy and loss comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Custom CNN", "VGG-16 (CIFAR-100 adapted)"],
    "Test Accuracy": [custom_test_accuracy, vgg_test_accuracy],
    "Test Loss": [custom_test_loss, vgg_test_loss]
})
comparison["Test Accuracy (%)"] = comparison["Test Accuracy"] * 100
display(comparison[["Model", "Test Accuracy (%)", "Test Loss"]].round(4))

best_model_name = comparison.loc[comparison["Test Accuracy"].idxmax(), "Model"]
print("Higher test accuracy:", best_model_name)

## 14. Results and conclusion

After running all cells, record the final test accuracies above. Compare the models using test accuracy, test loss, and the training/validation curves. A large gap between training and validation accuracy may indicate overfitting. Do not write assumed accuracy values before running the experiments.

## Submission checklist

- [x] CIFAR-100 dataset loaded and preprocessed for 100 classes
- [x] Custom CNN implemented from scratch
- [x] VGG-16-style architecture implemented in Keras
- [x] EarlyStopping and ModelCheckpoint callbacks used for both models
- [x] Model summaries displayed
- [x] Architecture diagrams generated for both models
- [x] Training/validation accuracy and loss curves plotted
- [x] Test accuracy and loss reported in a comparison table
- [ ] Run the notebook on Google Colab or Kaggle and save the outputs before submitting

**Note:** The provided rubric says “train-test curves”; in standard practice, training and validation curves are plotted during training, while the test set is evaluated once at the end. This avoids using test data to tune the models.

**Architecture note:** The VGG-16 feature extractor has the standard 13 convolutional layers. Its classifier head is adapted for CIFAR-100 and computational efficiency, so it is not an exact reproduction of the original VGG-16 4096-unit classifier.